# 03 · 고급 분석: 패널 고정효과  (모듈 4-A)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/03_panel_fe.ipynb)

> 고급 계량 분석도 AI 도움으로 **양쪽 도구에서** 할 수 있다. 패널 고정효과를 Python으로,
> 그리고 STATA로 — **둘 다 같은 결과**. 도구는 우열이 아니라 **환경(폐쇄망/외부망)**으로 고른다.

In [ ]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/sample_crs.csv")
df["log_disb"]  = np.log(df["USD_Disbursement"])
df["log_gdppc"] = np.log(df["RecipientGDPpc"])
df["log_pop"]   = np.log(df["RecipientPop"])
print("공여국-수원국 쌍 수:", df["pair_id"].nunique(), "· 연도:", df["Year"].nunique())

## 문제 — 쌍(공여국×수원국)의 고유 특성을 통제하려면?
역사·외교관계처럼 **잘 변하지 않는 쌍 고유 효과**를 빼고 순수 효과를 보려면 **고정효과**가 필요하다.

### (1) 통제 안 함 — 단순 회귀(pooled)

In [ ]:
pooled = smf.ols("log_disb ~ log_gdppc + log_pop", data=df).fit()
print("pooled  log_pop 계수 =", round(pooled.params["log_pop"], 3))

### (2) Python으로 고정효과 (쌍 효과 흡수)

In [ ]:
fe = smf.ols("log_disb ~ log_gdppc + log_pop + C(pair_id)", data=df).fit()
n_dummies = sum(c.startswith("C(pair_id)") for c in fe.params.index)
print("FE      log_pop 계수 =", round(fe.params["log_pop"], 3), "(쌍효과 흡수로 변화)")
print("추가된 쌍 더미 개수 =", n_dummies, "개 (쌍 고정효과)")

### (3) STATA에선 — 같은 분석
```stata
xtset pair_id
xtreg log_disb log_gdppc log_pop, fe vce(cluster pair_id)
```
- `xtreg, fe` 가 60개 쌍의 고정효과를 흡수하고, `vce(cluster ...)`로 강건표준오차까지. 위 Python의 `C(pair_id)`와 **동일한 모형**.
- (한 쌍·연도에 여러 분야 사업이 있어 **쌍 고정효과만** 선언.)
- 코드 → `stata/05_panel_fe.do`  ·  **`xtreg`는 폐쇄망에서 인터넷 없이 실행** ✅

---
✅ **포인트**: 고급 계량 모형도 이제 **양쪽에서** 가능 — AI가 코드를 짜준다.
도구는 *우열*이 아니라 **환경**으로 고른다(폐쇄망 현업은 STATA, 외부망 탐색·확장은 Python).